# Chien luoc: 
### Khi MA10 > MA20 va gia dong cua du doan > gia dong cua thuc te => Mua
### Khi MA10 < MA20 va gia dong cua du doan < gia dong cua thuc te => Sell

# Load du lieu len

In [ ]:
import sys
sys.path.append('../Common') # .. la ve thu muc truoc do

In [ ]:
# import Test

# trave = Test.Test.helloWorld()

# print(trave)

# symbol = 'STB.VN'
# from_date = '2024-10-01'
# to_date = '2024-10-16'
# interval = '1d'

# data = Test.Test.Hello(symbol, from_date, to_date, interval)

# data

# Check data trong Module con cua Common

In [ ]:
# import CommonYFinance # Import vao thu muc CommonYFinance

# symbol = 'ACB.VN'
# from_date = '2025-05-01'
# to_date = '2025-08-13'
# interval = '1d'

# data = CommonYFinance.CommonYFinance.loaddataYFinance(symbol, from_date, to_date, interval) # Module.Class.Ham

# # data.tail() # Lay 5 dong cuoi cung
# data

import MetaTrader5 as mt5
import TradeMT5
symbol = 'EURUSD.sml'
timeframe = mt5.TIMEFRAME_D1
limit = 100
data = TradeMT5.TradeMT5.getData_MT5(symbol, timeframe, limit)  
data

In [ ]:
# import CommonBinance
# import CommonBinanceDWH

# symbol = 'ETHUSDT'  # Binance symbol for Bitcoin
# from_date = '2025-05-01'
# to_date = '2025-06-21'
# interval = '1d'

# data = CommonBinanceDWH.CommonBinanceDWH.loaddataBinance_FromTo_Split(symbol, from_date, to_date, interval) 

# data

In [ ]:
data.info()

# Ve bieu do quan sat

In [ ]:
import matplotlib.pyplot as plt
# data.set_index('Datetime', inplace=True)

data['Close'].plot(title='Close Price')
plt.show()

# Tìm kiếm xu hướng, mùa vụ, hoặc các biến động không đồng đều

# Kiểm tra tính ổn định của chuỗi thời gian
### Noi sau

In [ ]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller # ADF

# Đọc dữ liệu (giả sử có sẵn file CSV)
# data = pd.read_csv("your_file.csv")

# Kiểm tra nếu cột 'Close' tồn tại
if 'Close' not in data.columns:
    raise ValueError("Cột 'Close' không tồn tại trong DataFrame!")

# Loại bỏ giá trị NaN nếu có
data = data.dropna(subset=['Close'])

# Kiểm tra chuỗi thời gian có đủ dữ liệu không
if len(data['Close']) < 10:
    raise ValueError("Dữ liệu quá ít để thực hiện kiểm định ADF!")

# Kiểm định ADF
result = adfuller(data['Close']) # Ham adfuller() se thuc hien viec kiem dinh ADF

# In kết quả
print('ADF Statistic: {:.6f}'.format(result[0]))
print('p-value: {:.6f}'.format(result[1]))
print('Critical Values:')
for key, value in result[4].items():
    print(f'\t{key}: {value:.3f}')

# Đưa ra kết luận
if result[1] > 0.05:
    print("Chuỗi có đơn vị gốc (không ổn định), cần phải differencing.")
else:
    print("Chuỗi ổn định, không cần biến đổi.")

# Ky thuant differencing



# Mo hinh

model = ARIMA(data['Close'], order=(5, 1, 0)): Lệnh này khởi tạo một mô hình ARIMA với dữ liệu đầu vào là cột 'Close' từ DataFrame data. Tham số order=(p, d, q) xác định cấu hình của mô hình, nơi:

p: số lượng lệnh tự hồi quy (AR terms). p = 5 nghĩa là mô hình sẽ xem xét 5 giá trị trước đó trong chuỗi để dự đoán giá trị hiện tại.
d: bậc của phép tích phân (I for Integrated), giúp làm cho chuỗi trở nên dừng. d = 1 chỉ ra rằng dữ liệu nên được chuyển đổi một lần (sử dụng sai phân) để đạt được tính dừng.
q: số lượng lệnh trung bình động (MA terms). q = 0 nghĩa là không có lệnh MA nào được sử dụng trong mô hình.

# Chay mo hinh du doan gia tuong lai

In [ ]:
data

# Gia su chung ta dua Order(5, 1, 0) de huan luyen (chung ta se tim gia tri toi uu cua p, d, q sau)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA # Khai bao

# Buoc 1: Khoi tao Xay dung mo hinh
model = ARIMA(data['Close'], order=(5, 1, 0))  # Ví dụ mô hình ARIMA với tham số (p, d, q)

# Buoc 2: Huan luyen Mo hinh
model_fit = model.fit() # Ham fit() se thuc hien viec toi uu hoa cac
# tham so cua mo hinh ARIMA dua tren du lieu dau vao

print(model_fit.summary())

# AIC cang nho thi mo hinh cang tot, tinh on dinh cao, mo hinh dang tin cay

# Du doan 1

In [ ]:
# Check data
data

In [12]:
import pandas as pd 
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import talib
import numpy as np

# Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
if 'Datetime' not in data.columns:
    print("Cột 'Datetime' không tồn tại trong DataFrame!")
else:
    data.set_index('Datetime', inplace=True)

print("=== PHÂN TÍCH VÀ CẢI THIỆN ARIMA ===")

# Bước 1: Kiểm tra tính dừng của dữ liệu
print("\n1. Kiểm tra tính dừng:")
result = adfuller(data['Close'])
print(f"ADF p-value: {result[1]:.6f}")

if result[1] > 0.05:
    print("❌ Dữ liệu chưa dừng - cần differencing")
    use_diff = True
else:
    print("✅ Dữ liệu đã dừng")
    use_diff = False

# Bước 2: Tạo các biến cần thiết
data['MA5'] = talib.SMA(data['Close'], timeperiod=5)
data['Close_diff'] = data['Close'].diff().dropna()

# Bước 3: Thử nhiều mô hình ARIMA
print("\n2. Thử các mô hình ARIMA:")

# models_to_try = [
#     # Sử dụng Close gốc
#     (data['Close'], (1, 1, 1), "Close - ARIMA(1,1,1)"),
#     (data['Close'], (2, 1, 2), "Close - ARIMA(2,1,2)"),
#     (data['Close'], (0, 1, 1), "Close - ARIMA(0,1,1)"),
#     (data['Close'], (1, 1, 0), "Close - ARIMA(1,1,0)"),
    
#     # Sử dụng Close_diff (nếu cần)
#     (data['Close_diff'], (1, 0, 1), "Close_diff - ARIMA(1,0,1)"),
#     (data['Close_diff'], (2, 0, 2), "Close_diff - ARIMA(2,0,2)"),
    
#     # Sử dụng MA5 (để so sánh)
#     (data['MA5'], (1, 1, 1), "MA5 - ARIMA(1,1,1)"),
#     (data['MA5'], (2, 0, 4), "MA5 - ARIMA(2,0,4) [Mô hình cũ]"),
# ]

models_to_try = [
    # Sử dụng Close gốc
    (data['Close'], (1, 0, 4), "Close - ARIMA(1,0,4)")   
]

best_model = None
best_aic = float('inf')
best_predictions = None
best_name = ""

for data_series, order, name in models_to_try:
    try:
        # Loại bỏ NaN
        clean_data = data_series.dropna()
        
        if len(clean_data) < 20:  # Cần ít nhất 20 quan sát
            print(f"❌ {name}: Dữ liệu quá ít ({len(clean_data)} quan sát)")
            continue
            
        model = ARIMA(clean_data, order=order)
        model_fit = model.fit()
        predictions = model_fit.forecast(steps=10)
        
        print(f"✅ {name}: AIC={model_fit.aic:.2f}")
        print(f"   Predictions: {predictions.values[:3]}...")
        
        if model_fit.aic < best_aic:
            best_aic = model_fit.aic
            best_model = model_fit
            best_predictions = predictions
            best_name = name
            
    except Exception as e:
        print(f"❌ {name}: Lỗi - {str(e)[:50]}...")

print(f"\n🏆 Mô hình tốt nhất: {best_name}")
print(f"AIC: {best_aic:.2f}")
print(f"Dự đoán: {best_predictions.values}")

# Bước 4: Trực quan hóa kết quả cải thiện
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Tạo subplot
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Giá gốc và dự đoán', 'Residuals'),
    vertical_spacing=0.1
)

# Plot 1: Giá gốc và dự đoán
fig.add_trace(
    go.Scatter(x=data.index, y=data['Close'], 
               mode='lines', name='Giá gốc', 
               line=dict(color='blue', width=2)),
    row=1, col=1
)

# Dự đoán với confidence interval
pred_dates = pd.date_range(start=data.index[-1], periods=11, freq='D')[1:]

fig.add_trace(
    go.Scatter(x=pred_dates, y=best_predictions, 
               mode='lines+markers', name='Dự đoán cải thiện', 
               line=dict(color='red', width=2, dash='dash')),
    row=1, col=1
)

# Plot 2: Residuals
residuals = best_model.resid
fig.add_trace(
    go.Scatter(x=data.index[1:len(residuals)+1], y=residuals, 
               mode='lines', name='Residuals', 
               line=dict(color='green')),
    row=2, col=1
)

fig.update_layout(
    height=600, 
    title_text=f"Dự đoán ARIMA cải thiện - {best_name}",
    showlegend=True
)

fig.show()

# Bước 5: Phân tích xu hướng
print(f"\n3. Phân tích xu hướng:")
current_price = data['Close'].iloc[-1]
predicted_price = best_predictions.iloc[-1]
change = predicted_price - current_price
change_pct = (change / current_price) * 100

print(f"Giá hiện tại: {current_price:.2f}")
print(f"Giá dự đoán (10 ngày): {predicted_price:.2f}")
print(f"Thay đổi: {change:+.2f} ({change_pct:+.2f}%)")

if change > 0:
    print("📈 Xu hướng: TĂNG")
elif change < 0:
    print("📉 Xu hướng: GIẢM")
else:
    print("➡️ Xu hướng: ỔN ĐỊNH")

Cột 'Datetime' không tồn tại trong DataFrame!
=== PHÂN TÍCH VÀ CẢI THIỆN ARIMA ===

1. Kiểm tra tính dừng:
ADF p-value: 0.033871
✅ Dữ liệu đã dừng

2. Thử các mô hình ARIMA:


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency B will be used.

c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency B will be used.

c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency B will be used.



✅ Close - ARIMA(1,0,4): AIC=-754.20
   Predictions: [1.16125301 1.16214028 1.16209821]...

🏆 Mô hình tốt nhất: Close - ARIMA(1,0,4)
AIC: -754.20
Dự đoán: [1.16125301 1.16214028 1.16209821 1.16223863 1.16234505 1.16243212
 1.16250337 1.16256167 1.16260937 1.1626484 ]


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals




3. Phân tích xu hướng:
Giá hiện tại: 1.16
Giá dự đoán (10 ngày): 1.16
Thay đổi: +0.00 (+0.09%)
📈 Xu hướng: TĂNG


# In ra predictions

In [ ]:
predictions

# Toi uu hoa mo hinh ARIMA voi p, d, q

In [ ]:
data.info()

In [ ]:
# import itertools # Import thu vien itertools, dung de tao ra cac cap tham so p, d, q
# p = d = q = range(0, 6) # Cho p tu 0 den 5, d tu 0 den 5, q tu 0 den 5
# pdq = list(itertools.product(p, d, q))

# print(pdq)
# print(len(pdq))

# Toi uu tham so p,d,q

In [ ]:
import itertools # Import thu vien itertools, dung de tao ra cac cap tham so p, d, q
# Tao ra cac cap tham so p, d, q
p = d = q = range(0, 6, 1) # Cho p tu 0 den 5, d tu 0 den 5, q tu 0 den 5
pdq = list(itertools.product(p, d, q)) # Tao ra cac cap tham so p, d, q
# So lan xuat hien pdq
print(pdq)
print(len(pdq)) # In ra so luong cap tham so

# Dua vao Close => Tinh p, d, q
# Tiep tuc, differencing: MA

In [ ]:
# import itertools

# p = d = q = range(0, 6, 1) # Cho p tu 0,1,2,3,4,5, d tu 0 den 5, q tu 0 den 5
# pdq = list(itertools.product(p, d, q)) # Tao ra cac cap tham so p, d, q
# print(pdq)

In [ ]:
import itertools
import statsmodels.api as sm
import pandas as pd
import talib

# Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
# data.set_index('Datetime', inplace=True) da dat o tren

# Xác định khoảng giá trị cho tham số p, d, q
p = d = q = range(0, 6, 1) # Cho p tu 0,1,2,3,4,5, d tu 0 den 5, q tu 0 den 5
pdq = list(itertools.product(p, d, q)) # Tao ra cac cap tham so p, d, q
# Chay 216 lan

best_aic = float("inf") # Khoi tao best_aic float('inf') là giá trị vô cực
best_pdq = None
best_model = None

# Diferencing: Cach 1: MA. Cach 2 Close_Diff
# data['Close_diff'] = data['Close'].diff().dropna()  # Close_diff la gia tri Close sau khi da diff
# data['MA'] = data['Close'].rolling(window=5).mean().dropna()  # Ví dụ dùng cửa sổ trượt kích thước 5
data["MA5"] = talib.MA(data['Close'], timeperiod=5).dropna() # Dùng thư viện TA-Lib

for param in pdq: # Duyet qua tat ca cac cap tham so p, d, q: 000
    print(param) # In ra cap tham so
    try:
        # model = sm.tsa.ARIMA(data['Close'], order=param) # (0,0,0) # (0,0,1) # (0,1,1)...(5,5,5)
        model = sm.tsa.ARIMA(data['MA5'], order=param) # (0,0,0) # (0,0,1) # (0,1,1)...(5,5,5)
        results = model.fit() # Moi lan dua vao ham fit() de huan luyen
        if results.aic <= best_aic: # AIC cang nho thi mo hinh cang tot, tinh on dinh cao, mo hinh dang tin cay
            best_aic = results.aic # Xac dinh best_aic la aic tra ve => aic nhat
            best_pdq = param
            best_model = results
    except:
        continue # Bo qua cac truong hop loi, chay lenh tiep theo cua for

print(f'Best ARIMA{best_pdq} AIC:{best_aic}')

In [ ]:
import itertools
import pandas as pd
import statsmodels.api as sm

# Tạo danh sách giá trị p, d, q
p = d = q = range(0, 6, 1)
pdq = list(itertools.product(p, d, q))

best_aic, best_bic, best_params = float('inf'), float('inf'), None
best_pdq = param
best_model = results

for param in pdq:
    try:
        model = sm.tsa.ARIMA(data["MA"], order=param)
        results = model.fit()
        
        if results.aic < best_aic and results.bic < best_bic:
            best_aic, best_bic, best_pdq = results.aic, results.bic, param
    except:
        continue

print(f'Best ARIMA{best_pdq} AIC:{best_aic} BIC:{best_bic}')

In [ ]:
# import pandas as pd
# from statsmodels.tsa.arima_model import ARIMA
# from sklearn.metrics import mean_squared_error

# # load data

# data.to_csv('vietcombank_stock_data.csv')
# series = pd.read_csv('vietcombank_stock_data.csv', header=0, index_col=0, parse_dates=True)

# # evaluate an ARIMA model for a given order (p,d,q)
# def evaluate_arima_model(order):
#     model = ARIMA(series, order=order)
#     model_fit = model.fit(disp=0)
#     mse = mean_squared_error(series, model_fit.fittedvalues)
#     return mse

# # evaluate combinations of p, d and q values for an ARIMA model
# def evaluate_models(p_values, d_values, q_values):
#     best_score, best_order = float("inf"), None
#     for p in p_values:
#         for d in d_values:
#             for q in q_values:
#                 order = (p,d,q)
#                 try:
#                     mse = evaluate_arima_model(order)
#                     if mse < best_score:
#                         best_score, best_order = mse, order
#                     print('ARIMA%s MSE=%.3f' % (order,mse))
#                 except:
#                     continue
#     print('Best ARIMA%s MSE=%.3f' % (best_order, best_score))

# # evaluate parameters
# p_values = [0, 1, 2, 4, 6, 8, 10]
# d_values = range(0, 3)
# q_values = range(0, 3)
# evaluate_models(p_values, d_values, q_values)

# Goi vao CommonArima


In [ ]:
# Goi vao CommonArima
import sys
sys.path.append('../Common')
import CommonArima

CommonArima.CommonArima.analysis_data(data)